In [1]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
#datos de ingreso
dia_ivd = '20260530'
dia_fvd = '20260531'

In [3]:
# Ruta de la carpeta que contiene los archivos de desglosados zonal
ruta_carpeta = 'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2026/Reporte_fin_semana'

# Fechas de inicio y fin para el filtro
fecha_inicio = f'{dia_ivd}'
fecha_fin = f'{dia_fvd}'

# Lista para almacenar los DataFrames
dataframes = []

# Recorrer todos los archivos en la carpeta
for nombre_archivo in os.listdir(ruta_carpeta):

    if nombre_archivo.endswith('_perdidos.csv'):

        # Extraer la fecha del nombre del archivo (YYYYMMDD)
        fecha_archivo = nombre_archivo[:8]

        # Verificar si la fecha está dentro del rango deseado
        if fecha_inicio <= fecha_archivo <= fecha_fin:

            ruta_archivo = os.path.join(ruta_carpeta, nombre_archivo)

            # Leer el archivo
            df = pd.read_csv(ruta_archivo, encoding='latin', sep=';')

            # Eliminar filas completamente vacías
            df = df.dropna(how='all')

            # Crear la columna Fecha a partir del nombre del archivo
            df['Fecha'] = pd.to_datetime(fecha_archivo, format='%Y%m%d')

            # Agregar a la lista
            dataframes.append(df)

# Verificar si se encontraron DataFrames
if dataframes:
    # Consolidar todos los DataFrames en uno solo
    perdidos = pd.concat(dataframes, ignore_index=True)
else:
    print("No se encontraron archivos para consolidar.")

# Verificar resultado
perdidos.head()


,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
0,2026-05-30,CE0610004,359,NILSON GARCIA SANCHEZ,508942.0,8.037380e+07,12:40:00,16:40:00,Gestion de Mtto,Varado en Vía,...,"40,662","21,324","19,338",2.0,RAUL FABIAN ROMERO MICAN,2026-05-30 19:49:18,NO,Z50-4031,PATIO LA Y,2026-05-30
1,2026-05-30,CE0610005,359,PAULO ALEXANDER PEREZ ACOSTA,509717.0,1.108765e+09,09:12:30,12:54:30,Gestion de Mtto,Varado en Vía,...,"40,662","8,756","31,906",2.0,ELICIA MONGUI DUARTE,2026-05-30 10:01:58,NO,Z50-2021,PATIO LA Y,2026-05-30
2,2026-05-30,CE0610010,359,BRAYAN ESTEBAN SANCHEZ TORRES,510665.0,1.000462e+09,15:07:00,18:47:00,Gestion de Mtto,Varado en Vía,...,"40,662","14,020","26,642",1.0,RAUL FABIAN ROMERO MICAN,2026-05-30 19:50:53,NO,Z50-4532,PATIO LA Y,2026-05-30
3,2026-05-30,CE0640001,DD204,JHON STIBEN CASTAÑEDA LEMUS,509249.0,1.000602e+09,05:16:30,05:52:30,Gestion de SV,Accidente,...,"10,904",0,"10,904",1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-05-30 05:36:10,NO,Z50-4312,PATIO TINTAL,2026-05-30
4,2026-05-30,CE0640001,DD204,JHON STIBEN CASTAÑEDA LEMUS,509249.0,1.000602e+09,05:54:00,06:32:00,Gestion de Mtto,Sin Movil para retomar,...,"10,904",0,"10,904",1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-05-30 05:56:23,NO,Z50-4312,PATIO TINTAL,2026-05-30


In [4]:
perdidos_filtrado = perdidos.copy()

perdidos_filtrado.head()

,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
0,2026-05-30,CE0610004,359,NILSON GARCIA SANCHEZ,508942.0,8.037380e+07,12:40:00,16:40:00,Gestion de Mtto,Varado en Vía,...,"40,662","21,324","19,338",2.0,RAUL FABIAN ROMERO MICAN,2026-05-30 19:49:18,NO,Z50-4031,PATIO LA Y,2026-05-30
1,2026-05-30,CE0610005,359,PAULO ALEXANDER PEREZ ACOSTA,509717.0,1.108765e+09,09:12:30,12:54:30,Gestion de Mtto,Varado en Vía,...,"40,662","8,756","31,906",2.0,ELICIA MONGUI DUARTE,2026-05-30 10:01:58,NO,Z50-2021,PATIO LA Y,2026-05-30
2,2026-05-30,CE0610010,359,BRAYAN ESTEBAN SANCHEZ TORRES,510665.0,1.000462e+09,15:07:00,18:47:00,Gestion de Mtto,Varado en Vía,...,"40,662","14,020","26,642",1.0,RAUL FABIAN ROMERO MICAN,2026-05-30 19:50:53,NO,Z50-4532,PATIO LA Y,2026-05-30
3,2026-05-30,CE0640001,DD204,JHON STIBEN CASTAÑEDA LEMUS,509249.0,1.000602e+09,05:16:30,05:52:30,Gestion de SV,Accidente,...,"10,904",0,"10,904",1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-05-30 05:36:10,NO,Z50-4312,PATIO TINTAL,2026-05-30
4,2026-05-30,CE0640001,DD204,JHON STIBEN CASTAÑEDA LEMUS,509249.0,1.000602e+09,05:54:00,06:32:00,Gestion de Mtto,Sin Movil para retomar,...,"10,904",0,"10,904",1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-05-30 05:56:23,NO,Z50-4312,PATIO TINTAL,2026-05-30


In [5]:
perdidos_filtrado["KMS PERDIDOS"] = (
    perdidos_filtrado["KMS PERDIDOS"]
    .astype(str)
    .str.replace(",", ".", regex=False)  # si usa coma decimal
    .str.replace(" ", "", regex=False)
    .str.strip()
)

perdidos_filtrado["KMS PERDIDOS"] = pd.to_numeric(
    perdidos_filtrado["KMS PERDIDOS"],
    errors="coerce"
)

perdidos_filtrado.head()

,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha
0,2026-05-30,CE0610004,359,NILSON GARCIA SANCHEZ,508942.0,8.037380e+07,12:40:00,16:40:00,Gestion de Mtto,Varado en Vía,...,"40,662","21,324",19.338,2.0,RAUL FABIAN ROMERO MICAN,2026-05-30 19:49:18,NO,Z50-4031,PATIO LA Y,2026-05-30
1,2026-05-30,CE0610005,359,PAULO ALEXANDER PEREZ ACOSTA,509717.0,1.108765e+09,09:12:30,12:54:30,Gestion de Mtto,Varado en Vía,...,"40,662","8,756",31.906,2.0,ELICIA MONGUI DUARTE,2026-05-30 10:01:58,NO,Z50-2021,PATIO LA Y,2026-05-30
2,2026-05-30,CE0610010,359,BRAYAN ESTEBAN SANCHEZ TORRES,510665.0,1.000462e+09,15:07:00,18:47:00,Gestion de Mtto,Varado en Vía,...,"40,662","14,020",26.642,1.0,RAUL FABIAN ROMERO MICAN,2026-05-30 19:50:53,NO,Z50-4532,PATIO LA Y,2026-05-30
3,2026-05-30,CE0640001,DD204,JHON STIBEN CASTAÑEDA LEMUS,509249.0,1.000602e+09,05:16:30,05:52:30,Gestion de SV,Accidente,...,"10,904",0,10.904,1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-05-30 05:36:10,NO,Z50-4312,PATIO TINTAL,2026-05-30
4,2026-05-30,CE0640001,DD204,JHON STIBEN CASTAÑEDA LEMUS,509249.0,1.000602e+09,05:54:00,06:32:00,Gestion de Mtto,Sin Movil para retomar,...,"10,904",0,10.904,1.0,JULIAN ANDRES SANCHEZ RESTREPO,2026-05-30 05:56:23,NO,Z50-4312,PATIO TINTAL,2026-05-30


In [6]:
analisis_patio = (
    perdidos_filtrado.groupby("PATIO")
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count"),
        promedio_km_por_evento=("KMS PERDIDOS", "mean")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_patio.head()

,total_km_perdidos,eventos,promedio_km_por_evento
PATIO,,,
PATIO TINTAL,2151.578,64,33.618406
PATIO TINTAL 2,1013.482,54,18.768185
LA VERBENA_GM,981.773,34,28.875676
GMOVILTRONCAL,291.881,14,20.848643
HIBRIDOS,93.226,2,46.613000


In [7]:
analisis_patio_ruta = (
    perdidos_filtrado.groupby(["PATIO", "RUTA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_patio_ruta.head()

total_km_perdidos  eventos
PATIO          RUTA                             
PATIO TINTAL   DL219            525.434       11
               DH209            372.600        8
PATIO TINTAL 2 SE14             269.948        8
LA VERBENA_GM  740              243.118        8
PATIO TINTAL   E25              239.098        8

In [8]:
analisis_detallado = (
    perdidos_filtrado.groupby(
        ["FECHA", "PATIO", "RUTA", "MOTIVO", "CAUSA"]
    )
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_detallado.head(20)

total_km_perdidos  \
FECHA      PATIO          RUTA              MOTIVO                       CAUSA                                          
2026-05-31 PATIO TINTAL   DH209             Gestion del Operador         Operador en otro servicio            186.300   
           LA VERBENA_GM  740               Gestion del Operador         Operador no se presenta              183.471   
           PATIO TINTAL   DL219             Gestion del Operador         Operador en otro servicio            177.291   
                          DH209             Gestion del Operador         Operador no se presenta              139.725   
2026-05-30 PATIO TINTAL   E25               Gestion del Operador         Operador no se presenta              124.368   
2026-05-31 PATIO TINTAL   806               Gestion del Operador         Operador en otro servicio            121.237   
                          DL219             Gestion de Mtto              Varado en Vía                        121.056   
           PATIO TINTAL 2 16-5 VILLA AMALIA Gestion de Centro de Control Retraso por CongestiÃ³n              118.594   
           PATIO TINTAL   DA213             Gestion del Operador         Operador en otro servicio            109.431   
           HIBRIDOS       M86-K86           Gestion de Centro de Control Cambio de Linea                       93.226   
           PATIO TINTAL 2 DH216             Gestion del Operador         Operador no se presenta               87.136   
                          539               Gestion del Operador         Operador no se presenta               85.800   
           LA VERBENA_GM  142               Gestion del Operador         Operador no se presenta               78.432   
2026-05-30 PATIO LA Y     359               Gestion de Mtto              Varado en Vía                         77.886   
2026-05-31 LA VERBENA_GM  BD237             Gestion del Operador         Operador en otro servicio             77.370   
                          577               Gestion del Operador         Operador no se presenta               76.961   
2026-05-30 PATIO TINTAL 2 SE14              Gestion de Centro de Control Retraso por CongestiÃ³n               73.818   
2026-05-31 PATIO TINTAL 2 SE14              Gestion del Operador         Operador no se presenta               73.250   
           LA VERBENA_GM  576               Gestion del Operador         Operador no se presenta               66.469   
           PATIO TINTAL   402               Gestion del Operador         Operador llega retrasado              64.199   

                                                                                                    eventos  
FECHA      PATIO          RUTA              MOTIVO                       CAUSA                               
2026-05-31 PATIO TINTAL   DH209             Gestion del Operador         Operador en otro servicio        4  
           LA VERBENA_GM  740               Gestion del Operador         Operador no se presenta          3  
           PATIO TINTAL   DL219             Gestion del Operador         Operador en otro servicio        3  
                          DH209             Gestion del Operador         Operador no se presenta          3  
2026-05-30 PATIO TINTAL   E25               Gestion del Operador         Operador no se presenta          4  
2026-05-31 PATIO TINTAL   806               Gestion del Operador         Operador en otro servicio        4  
                          DL219             Gestion de Mtto              Varado en Vía                    4  
           PATIO TINTAL 2 16-5 VILLA AMALIA Gestion de Centro de Control Retraso por CongestiÃ³n          7  
           PATIO TINTAL   DA213             Gestion del Operador         Operador en otro servicio        3  
           HIBRIDOS       M86-K86           Gestion de Centro de Control Cambio de Linea                  2  
           PATIO TINTAL 2 DH216             Gestion del Operador         Operador no se presenta          2  
       

In [9]:
top5_rutas = (
    perdidos_filtrado.groupby("RUTA")
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top5_rutas

,total_km_perdidos
RUTA,
DL219,525.434
DH209,372.600
SE14,269.948
740,243.118
E25,239.098


In [10]:
top5_patio_ruta = (
    perdidos_filtrado.groupby(["PATIO", "RUTA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top5_patio_ruta

total_km_perdidos
PATIO          RUTA                    
PATIO TINTAL   DL219            525.434
               DH209            372.600
PATIO TINTAL 2 SE14             269.948
LA VERBENA_GM  740              243.118
PATIO TINTAL   E25              239.098

In [11]:
top5_causa = (
    perdidos_filtrado.groupby(["MOTIVO", "CAUSA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top5_causa

total_km_perdidos
MOTIVO                       CAUSA                                       
Gestion del Operador         Operador no se presenta             1377.671
                             Operador en otro servicio            884.667
Gestion de Mtto              Varado en Vía                        568.961
Gestion de Centro de Control Retraso por CongestiÃ³n              562.941
Gestion del Operador         Operador llega retrasado             251.164

In [12]:
top5_patio_ruta = (
    perdidos_filtrado.groupby(["PATIO", "RUTA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top5_patio_ruta

total_km_perdidos
PATIO          RUTA                    
PATIO TINTAL   DL219            525.434
               DH209            372.600
PATIO TINTAL 2 SE14             269.948
LA VERBENA_GM  740              243.118
PATIO TINTAL   E25              239.098

In [13]:
# Convertimos el índice a lista de tuplas
top5_keys = top5_patio_ruta.index.tolist()

detalle_top5 = (
    perdidos_filtrado[
        perdidos_filtrado.set_index(["PATIO", "RUTA"]).index.isin(top5_keys)
    ]
    .groupby(["PATIO", "RUTA", "FECHA", "MOTIVO", "CAUSA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

detalle_top5

total_km_perdidos  \
PATIO          RUTA  FECHA      MOTIVO                       CAUSA                                                   
PATIO TINTAL   DH209 2026-05-31 Gestion del Operador         Operador en otro servicio                     186.300   
LA VERBENA_GM  740   2026-05-31 Gestion del Operador         Operador no se presenta                       183.471   
PATIO TINTAL   DL219 2026-05-31 Gestion del Operador         Operador en otro servicio                     177.291   
               DH209 2026-05-31 Gestion del Operador         Operador no se presenta                       139.725   
               E25   2026-05-30 Gestion del Operador         Operador no se presenta                       124.368   
               DL219 2026-05-31 Gestion de Mtto              Varado en Vía                                 121.056   
PATIO TINTAL 2 SE14  2026-05-30 Gestion de Centro de Control Retraso por CongestiÃ³n                        73.818   
                     2026-05-31 Gestion del Operador         Operador no se presenta                        73.250   
PATIO TINTAL   E25   2026-05-31 Gestion del Operador         Operador en otro servicio                      62.184   
               DL219 2026-05-30 Gestion del Operador         Operador con novedad varado en via             59.097   
                                                             Operador llega retrasado                       59.097   
                                                             Operador no se presenta                        59.097   
                                Gestion de Mtto              Falla Sirci en vía                             49.796   
               DH209 2026-05-30 Gestion del Operador         Operador en otro servicio                      46.575   
LA VERBENA_GM  740   2026-05-30 Gestion de SV                Accidente                                      44.501   
PATIO TINTAL 2 SE14  2026-05-31 Gestion de Mtto              Movil no despachado                            36.909   
                                Gestion de Centro de Control Retraso por CongestiÃ³n                        36.341   
PATIO TINTAL   E25   2026-05-31 Gestion del Operador         Operador no se presenta                        31.792   
PATIO TINTAL 2 SE14  2026-05-31 Gestion de Mtto              Varado en Vía                                  31.676   
PATIO TINTAL   E25   2026-05-31 Gestion de SV                Accidente                                      20.754   
PATIO TINTAL 2 SE14  2026-05-31 Gestion de Mtto              Falla Sirci en vía                             17.954   
LA VERBENA_GM  740   2026-05-30 Gestion de Mtto              Movil llega tarde - Incorporación               5.951   
                                                             Falla Sirci en vía                              4.911   
                                                             Varado en Vía                                   4.284   

                                                                                                 eventos  
PATIO          RUTA  FECHA      MOTIVO                       CAUSA                                        
PATIO TINTAL   DH209 2026-05-31 Gestion del Operador         Operador en otro servicio                 4  
LA VERBENA_GM  740   2026-05-31 Gestion del Operador         Operador no se presenta                   3  
PATIO TINTAL   DL219 2026-05-31 Gestion del Operador         Operador en otro servicio                 3  
               DH209 2026-05-31 Gestion del Operador         Operador no se presenta                   3  
               E25   2026-05-30 Gestion del Operador         Operador no se presenta                   4  
               DL219 2026-05-31 Gestion de Mtto              Varado en Vía                             4  
PATIO TINTAL 2 SE14  2026-05-30 Gestion de Centro de Control Retraso por CongestiÃ³n                   2  
                     2026-05-31 Gestion del Operador         Oper

In [14]:
top_evento_por_patio = (
    perdidos_filtrado
    .groupby(["PATIO", "RUTA", "FECHA", "MOTIVO", "CAUSA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .reset_index()
)

top_evento_por_patio

,PATIO,RUTA,FECHA,MOTIVO,CAUSA,total_km_perdidos
0,GMOVILTRONCAL,B46 - Portal Norte,2026-05-30,Gestion de Mtto,Varado en Vía,31.864
1,GMOVILTRONCAL,C15-H15,2026-05-30,Gestion de Mtto,Sin Movil para retomar,28.717
2,GMOVILTRONCAL,C15-H15,2026-05-30,Gestion de Mtto,Varado en Vía,28.717
3,GMOVILTRONCAL,C15-H15,2026-05-31,Gestion de Mtto,Varado en Vía,37.150
4,GMOVILTRONCAL,C25-L25,2026-05-30,Gestion de Mtto,Varado en Vía,37.056
...,...,...,...,...,...,...
103,PATIO TINTAL 2,SE14,2026-05-31,Gestion de Centro de Control,Retraso por CongestiÃ³n,36.341
104,PATIO TINTAL 2,SE14,2026-05-31,Gestion de Mtto,Falla Sirci en vía,17.954
105,PATIO TINTAL 2,SE14,2026-05-31,Gestion de Mtto,Movil no despachado,36.909
106,PATIO TINTAL 2,SE14,2026-05-31,Gestion de Mtto,Varado en Vía,31.676


In [15]:
top_por_patio = (
    top_evento_por_patio
    .sort_values("total_km_perdidos", ascending=False)
    .groupby("PATIO")
    .head(1)   # toma el más alto de cada patio
)

top_por_patio

,PATIO,RUTA,FECHA,MOTIVO,CAUSA,total_km_perdidos
63,PATIO TINTAL,DH209,2026-05-31,Gestion del Operador,Operador en otro servicio,186.300
20,LA VERBENA_GM,740,2026-05-31,Gestion del Operador,Operador no se presenta,183.471
89,PATIO TINTAL 2,16-5 VILLA AMALIA,2026-05-31,Gestion de Centro de Control,Retraso por CongestiÃ³n,118.594
9,HIBRIDOS,M86-K86,2026-05-31,Gestion de Centro de Control,Cambio de Linea,93.226
33,PATIO LA Y,359,2026-05-30,Gestion de Mtto,Varado en Vía,77.886
5,GMOVILTRONCAL,C25-L25,2026-05-31,Gestion de Mtto,Varado en Vía,63.671


In [16]:
top5_patio_evento = (
    top_por_patio
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top5_patio_evento

,PATIO,RUTA,FECHA,MOTIVO,CAUSA,total_km_perdidos
63,PATIO TINTAL,DH209,2026-05-31,Gestion del Operador,Operador en otro servicio,186.300
20,LA VERBENA_GM,740,2026-05-31,Gestion del Operador,Operador no se presenta,183.471
89,PATIO TINTAL 2,16-5 VILLA AMALIA,2026-05-31,Gestion de Centro de Control,Retraso por CongestiÃ³n,118.594
9,HIBRIDOS,M86-K86,2026-05-31,Gestion de Centro de Control,Cambio de Linea,93.226
33,PATIO LA Y,359,2026-05-30,Gestion de Mtto,Varado en Vía,77.886


In [17]:
top_evento_por_fecha = (
    perdidos_filtrado
    .groupby(["FECHA", "PATIO", "RUTA", "MOTIVO", "CAUSA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .reset_index()
)

top_evento_por_fecha.head(10)

,FECHA,PATIO,RUTA,MOTIVO,CAUSA,total_km_perdidos
0,2026-05-30,GMOVILTRONCAL,B46 - Portal Norte,Gestion de Mtto,Varado en Vía,31.864
1,2026-05-30,GMOVILTRONCAL,C15-H15,Gestion de Mtto,Sin Movil para retomar,28.717
2,2026-05-30,GMOVILTRONCAL,C15-H15,Gestion de Mtto,Varado en Vía,28.717
3,2026-05-30,GMOVILTRONCAL,C25-L25,Gestion de Mtto,Varado en Vía,37.056
4,2026-05-30,GMOVILTRONCAL,H54 - K54,Gestion de Mtto,Varado en Vía,12.782
5,2026-05-30,GMOVILTRONCAL,K43 - G43,Gestion de Mtto,Varado en Vía,17.957
6,2026-05-30,GMOVILTRONCAL,L10 - K10,Gestion de Centro de Control,Retraso por CongestiÃ³n,33.967
7,2026-05-30,LA VERBENA_GM,576,Gestion de Mtto,Movil llega tarde - Incorporación,4.605
8,2026-05-30,LA VERBENA_GM,577,Gestion de Mtto,Sin Movil para retomar,37.292
9,2026-05-30,LA VERBENA_GM,577,Gestion del Operador,Operador no se presenta,39.669


In [18]:
top_por_fecha = (
    top_evento_por_fecha
    .sort_values("total_km_perdidos", ascending=False)
    .groupby("FECHA")
    .head(1)
    .sort_values("FECHA")
)

top_por_fecha

,FECHA,PATIO,RUTA,MOTIVO,CAUSA,total_km_perdidos
43,2026-05-30,PATIO TINTAL,E25,Gestion del Operador,Operador no se presenta,124.368
84,2026-05-31,PATIO TINTAL,DH209,Gestion del Operador,Operador en otro servicio,186.300


In [19]:
top_eventos = (
    perdidos_filtrado
    .groupby(["FECHA", "PATIO", "RUTA", "MOTIVO", "CAUSA"])
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum")
    )
    .reset_index()
)

In [20]:
top5_por_fecha = (
    top_eventos
    .sort_values(["FECHA", "total_km_perdidos"], ascending=[True, False])
    .groupby("FECHA")
    .head(5)
)

top5_por_fecha

,FECHA,PATIO,RUTA,MOTIVO,CAUSA,total_km_perdidos
43,2026-05-30,PATIO TINTAL,E25,Gestion del Operador,Operador no se presenta,124.368
21,2026-05-30,PATIO LA Y,359,Gestion de Mtto,Varado en Vía,77.886
59,2026-05-30,PATIO TINTAL 2,SE14,Gestion de Centro de Control,Retraso por CongestiÃ³n,73.818
53,2026-05-30,PATIO TINTAL 2,402,Gestion de SV,Accidente,63.227
40,2026-05-30,PATIO TINTAL,DL219,Gestion del Operador,Operador con novedad varado en via,59.097
84,2026-05-31,PATIO TINTAL,DH209,Gestion del Operador,Operador en otro servicio,186.300
66,2026-05-31,LA VERBENA_GM,740,Gestion del Operador,Operador no se presenta,183.471
87,2026-05-31,PATIO TINTAL,DL219,Gestion del Operador,Operador en otro servicio,177.291
85,2026-05-31,PATIO TINTAL,DH209,Gestion del Operador,Operador no se presenta,139.725
74,2026-05-31,PATIO TINTAL,806,Gestion del Operador,Operador en otro servicio,121.237


In [21]:
falta_movil_tarde = perdidos_filtrado[
    (perdidos_filtrado["MOTIVO"] == "Falta de Movil") &
    (perdidos_filtrado["CAUSA"] == "Movil llega tarde - Incorporación")
]

falta_movil_tarde.head()

,FECHA,SERVICIO VEHICULO,RUTA,OPERADOR,CODIGO,CEDULA,HORA INICIO,HORA FINAL,MOTIVO,CAUSA,...,KMS PLANIFICADOS,KMS REALIZADOS,KMS PERDIDOS,PARTE DE TRABAJO,USUARIO REGISTRO,FECHA REGISTRO,REPORTADO A TRANSMILENIO,VEHICULO,PATIO,Fecha


In [22]:
analisis_fechas = (
    falta_movil_tarde
    .groupby("FECHA")
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("FECHA")
)

analisis_fechas

,total_km_perdidos,eventos
FECHA,,


In [23]:
analisis_patio = (
    falta_movil_tarde
    .groupby("PATIO")
    .agg(
        total_km_perdidos=("KMS PERDIDOS", "sum"),
        eventos=("KMS PERDIDOS", "count")
    )
    .sort_values("total_km_perdidos", ascending=False)
)

analisis_patio

,total_km_perdidos,eventos
PATIO,,


In [24]:
total_general = perdidos_filtrado["KMS PERDIDOS"].sum()

In [25]:
por_motivo = (
    perdidos_filtrado
    .groupby("MOTIVO")
    .agg(total_km_perdidos=("KMS PERDIDOS", "sum"))
    .reset_index()
)

por_motivo["porcentaje"] = (
    por_motivo["total_km_perdidos"] / total_general
) * 100

por_motivo = por_motivo.sort_values("total_km_perdidos", ascending=False)

por_motivo

,MOTIVO,total_km_perdidos,porcentaje
3,Gestion del Operador,2661.190,57.728643
1,Gestion de Mtto,1032.399,22.395618
0,Gestion de Centro de Control,689.008,14.946508
2,Gestion de SV,227.229,4.929232


In [26]:
por_motivo = (
    perdidos_filtrado
    .groupby("CAUSA")
    .agg(total_km_perdidos=("KMS PERDIDOS", "sum"))
    .reset_index()
)

por_motivo["porcentaje"] = (
    por_motivo["total_km_perdidos"] / total_general
) * 100

por_motivo = por_motivo.sort_values("total_km_perdidos", ascending=False)

por_motivo

,CAUSA,total_km_perdidos,porcentaje
13,Operador no se presenta,1377.671,29.885531
9,Operador en otro servicio,884.667,19.190898
16,Varado en Vía,568.961,12.342353
14,Retraso por CongestiÃ³n,562.941,12.211762
11,Operador llega retrasado,251.164,5.448449
0,Accidente,227.229,4.929232
15,Sin Movil para retomar,165.199,3.583628
3,Falla Sirci en vía,129.064,2.799759
4,Movil llega tarde - Incorporación,93.581,2.030033
2,Cambio de Linea,93.226,2.022332


In [27]:
total_general = perdidos_filtrado["KMS PERDIDOS"].sum()

por_motivo = (
    perdidos_filtrado
    .groupby(["FECHA", "CAUSA"])
    .agg(total_km_perdidos=("KMS PERDIDOS", "sum"))
    .reset_index()
)

por_motivo["porcentaje"] = (
    por_motivo["total_km_perdidos"] / total_general
) * 100

por_motivo = por_motivo.sort_values(
    ["FECHA", "total_km_perdidos"],
    ascending=[True, False]
)

por_motivo

,FECHA,CAUSA,total_km_perdidos,porcentaje
12,2026-05-30,Operador no se presenta,320.726,6.957443
15,2026-05-30,Varado en Vía,278.280,6.036670
0,2026-05-30,Accidente,206.475,4.479019
13,2026-05-30,Retraso por CongestiÃ³n,173.022,3.753330
14,2026-05-30,Sin Movil para retomar,136.288,2.956467
10,2026-05-30,Operador llega retrasado,126.155,2.736654
2,2026-05-30,Falla Sirci en vía,111.110,2.410286
3,2026-05-30,Movil llega tarde - Incorporación,93.581,2.030033
8,2026-05-30,Operador en otro servicio,80.505,1.746378
6,2026-05-30,Operador con novedad varado en via,59.097,1.281979


In [28]:
tabla_km_fecha = (
    perdidos_filtrado
    .groupby(["FECHA", "CAUSA"])
    .agg(total_km_perdidos=("KMS PERDIDOS", "sum"))
    .reset_index()
    .sort_values(["FECHA", "total_km_perdidos"], ascending=[True, False])
)

tabla_km_fecha

,FECHA,CAUSA,total_km_perdidos
12,2026-05-30,Operador no se presenta,320.726
15,2026-05-30,Varado en Vía,278.280
0,2026-05-30,Accidente,206.475
13,2026-05-30,Retraso por CongestiÃ³n,173.022
14,2026-05-30,Sin Movil para retomar,136.288
10,2026-05-30,Operador llega retrasado,126.155
2,2026-05-30,Falla Sirci en vía,111.110
3,2026-05-30,Movil llega tarde - Incorporación,93.581
8,2026-05-30,Operador en otro servicio,80.505
6,2026-05-30,Operador con novedad varado en via,59.097


In [29]:
total_general = perdidos_filtrado["KMS PERDIDOS"].sum()

tabla_porcentaje_causa = (
    perdidos_filtrado
    .groupby("CAUSA")
    .agg(total_km_perdidos=("KMS PERDIDOS", "sum"))
    .reset_index()
)

tabla_porcentaje_causa["porcentaje"] = (
    tabla_porcentaje_causa["total_km_perdidos"] / total_general
) * 100

tabla_porcentaje_causa = tabla_porcentaje_causa.sort_values(
    "total_km_perdidos",
    ascending=False
)

tabla_porcentaje_causa

,CAUSA,total_km_perdidos,porcentaje
13,Operador no se presenta,1377.671,29.885531
9,Operador en otro servicio,884.667,19.190898
16,Varado en Vía,568.961,12.342353
14,Retraso por CongestiÃ³n,562.941,12.211762
11,Operador llega retrasado,251.164,5.448449
0,Accidente,227.229,4.929232
15,Sin Movil para retomar,165.199,3.583628
3,Falla Sirci en vía,129.064,2.799759
4,Movil llega tarde - Incorporación,93.581,2.030033
2,Cambio de Linea,93.226,2.022332


In [30]:
tabla_base = (
    perdidos_filtrado
    .groupby(["FECHA", "CAUSA"])
    .agg(total_km_perdidos=("KMS PERDIDOS", "sum"))
    .reset_index()
)

In [31]:
top_por_fecha = (
    tabla_base
    .sort_values(["FECHA", "total_km_perdidos"], ascending=[True, False])
    .groupby("FECHA")
    .head(5)
)

top_por_fecha

,FECHA,CAUSA,total_km_perdidos
12,2026-05-30,Operador no se presenta,320.726
15,2026-05-30,Varado en Vía,278.280
0,2026-05-30,Accidente,206.475
13,2026-05-30,Retraso por CongestiÃ³n,173.022
14,2026-05-30,Sin Movil para retomar,136.288
22,2026-05-31,Operador no se presenta,1056.945
20,2026-05-31,Operador en otro servicio,804.162
23,2026-05-31,Retraso por CongestiÃ³n,389.919
25,2026-05-31,Varado en Vía,290.681
21,2026-05-31,Operador llega retrasado,125.009


In [32]:
top_21 = (
    tabla_base[tabla_base["FECHA"] == "2026-02-21"]
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top_22 = (
    tabla_base[tabla_base["FECHA"] == "2026-02-22"]
    .sort_values("total_km_perdidos", ascending=False)
    .head(5)
)

top_21
top_22

,FECHA,CAUSA,total_km_perdidos
